# 🏢 Hierarchical Customer Support RL — GRPO Training
**Meta × PyTorch × Scaler OpenEnv Hackathon**

Trains `Qwen2.5-1.5B-Instruct` with **GRPO** on the `curriculum_basic` task (L1-only, easiest stage).

**Steps:**
1. Install Unsloth + deps
2. Configure env URL + API key
3. Load model with LoRA
4. Baseline evaluation (before training)
5. GRPO training loop
6. Post-training evaluation
7. Reward / loss curves + before/after comparison

## 1️⃣ Install Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install httpx matplotlib numpy
print('✅ Done')

## 2️⃣ Configuration

Set `ENV_URL` to your **ngrok tunnel** or **HF Space URL** pointing at the environment server (port 7860).

In [ ]:
# ── EDIT THESE ────────────────────────────────────────────────────────────────
ENV_URL  = "https://YOUR-NGROK-URL.ngrok-free.app"  # ← paste your ngrok/HF Space URL
API_KEY  = "meta_hack_2026"
MODEL    = "unsloth/Qwen2.5-1.5B-Instruct"          # ~1 GB at 4-bit, fits on T4

# Training knobs (safe defaults for Colab T4)
TASK           = "curriculum_basic"   # L1-only — no supervisor, easiest task
TOTAL_STEPS    = 40                   # increase for better results (≥100 recommended)
GROUP_SIZE     = 4                    # rollouts per GRPO group
LORA_R         = 16
LEARNING_RATE  = 5e-5
MAX_NEW_TOKENS = 150
N_EVAL         = 3                    # episodes per eval run

import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f'Config ready — ENV: {ENV_URL}  MODEL: {MODEL}  STEPS: {TOTAL_STEPS}')

## 3️⃣ Verify Environment Connection

In [ ]:
import httpx, json, re, time

HEADERS = {"X-API-Key": API_KEY, "Content-Type": "application/json"}

try:
    r = httpx.get(f"{ENV_URL}/health", timeout=15)
    r.raise_for_status()
    print("✅ Environment reachable:", r.json())
except Exception as e:
    raise ConnectionError(
        f"\n❌ Cannot reach environment at {ENV_URL}\n"
        f"   Error: {e}\n\n"
        f"   Fix: update ENV_URL in cell 2 to your ngrok tunnel or HF Space URL."
    )

## 4️⃣ Load Model + LoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LEN = 2048
print(f"Loading {MODEL} (4-bit)...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id

# Frozen reference model for KL divergence penalty
ref_model, _ = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True,
)
for p in ref_model.parameters():
    p.requires_grad_(False)
ref_model.eval()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ Model ready — trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")

## 5️⃣ Prompt Builder, Action Parser & Env Client

In [ ]:
# ── System prompt (matches train/prompt_builder.py in the repo) ───────────────
SYSTEM_PROMPT = """CRITICAL RULE — READ FIRST:
If priority is "critical":
1. Step 1: ALWAYS output a "respond" action first to acknowledge the customer.
2. Step 2: Use "escalate" action. Reason MUST include an urgency keyword (e.g. "SLA breach").

You are an AI customer support agent resolving support tickets.

SCORING:
- TONE (20%): Be warm and empathetic. Cold replies are penalised heavily.
- EFFICIENCY (20%): Resolve faster = higher score.
- ACCURACY (20%): Gather ALL items in "Unresolved issues" before closing.
- RESOLUTION (40%): Use clear resolution language. Output >60 characters to avoid terse penalties.

ACTION TYPES — output exactly one per step:
- "respond"      → send a message to the customer    → requires: "message"
- "request_info" → ask for missing information        → requires: "message"
- "close"        → close the ticket as resolved       → requires: "message"
- "escalate"     → hand off to a specialist           → requires: "reason" (NOT message)

HARD RULES:
- NEVER close if "Unresolved issues" is non-empty.
- NEVER escalate low/medium priority tickets.
- Keep messages 60-300 characters.

OUTPUT FORMAT — return ONLY this JSON, no code fences, no preamble:
{"action_type": "...", "message": "..."}    ← for respond / request_info / close
{"action_type": "escalate", "reason": "..."} ← for escalate"""


def build_prompt(obs: dict) -> str:
    history = "\n".join(
        f"{m['role'].upper()}: {m['content']}"
        for m in obs.get("conversation_history", [])
    )
    unresolved = ", ".join(obs.get("unresolved_issues", [])) or "none"
    ctx = (
        f"Ticket: {obs['subject']}\n"
        f"Category: {obs['category']} | Priority: {obs['priority']} | "
        f"Step: {obs['step']}/{obs['max_steps']}\n"
        f"Sentiment: {obs['customer_sentiment']:.2f}\n"
        f"Unresolved: {unresolved}\n"
    )
    if obs.get("environment_event"):
        ctx += f"\n⚠️ EVENT: {obs['environment_event']}\n"
    ctx += f"\nConversation:\n{history}\n\nOutput JSON only."
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": ctx},
    ]
    try:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )


# Role-aware valid action sets
_ROLE_ACTIONS = {
    "support_agent": {"respond", "escalate", "close", "request_info"},
    "supervisor":    {"supervisor_approve", "supervisor_reject",
                      "supervisor_feedback", "supervisor_escalate"},
    "manager":       {"manager_override", "manager_resolve", "manager_send_back"},
}
_FALLBACK = {"action_type": "respond",
             "message": "I apologize for the inconvenience. Let me look into this for you right away."}


def parse_action(text: str, active_role: str = "support_agent") -> dict:
    """Parse model output into an action dict; return fallback on failure."""
    # Strip Qwen3 chain-of-thought blocks
    text = re.sub(r"<think>[\s\S]*?</think>", "", text, flags=re.IGNORECASE).strip()
    # Strip markdown code fences
    text = re.sub(r"```[a-z]*\n?", "", text).strip()
    m = re.search(r"\{[\s\S]*?\}", text)
    if not m:
        return dict(_FALLBACK)
    try:
        action = json.loads(m.group(0))
        allowed = _ROLE_ACTIONS.get(active_role, _ROLE_ACTIONS["support_agent"])
        if action.get("action_type") not in allowed:
            return dict(_FALLBACK)
        return action
    except Exception:
        return dict(_FALLBACK)


# ── Environment helpers ───────────────────────────────────────────────────────
def env_reset(task: str):
    r = httpx.post(f"{ENV_URL}/reset", params={"task": task}, headers=HEADERS, timeout=30)
    r.raise_for_status()
    d = r.json()
    return d["session_id"], d["observation"]


def env_step(session_id: str, action: dict):
    r = httpx.post(
        f"{ENV_URL}/step",
        params={"session_id": session_id},
        content=json.dumps(action),
        headers=HEADERS,
        timeout=60,
    )
    r.raise_for_status()
    return r.json()


print("✅ Prompt builder, action parser, and env client ready")

## 6️⃣ Rollout & Log-Prob Utilities

In [ ]:
def generate_text(prompt: str, do_sample: bool = True) -> str:
    """Generate a completion for the given prompt."""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=MAX_SEQ_LEN - MAX_NEW_TOKENS).to("cuda")
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=0.7 if do_sample else 1.0,
            top_p=0.9 if do_sample else 1.0,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(ids[0, prompt_len:], skip_special_tokens=True)


def get_log_probs(mdl, prompt: str, completion: str) -> torch.Tensor:
    """
    Return per-token log-probs of `completion` under `mdl`.
    Shape: (n_completion_tokens,)
    """
    full_text = prompt + completion
    enc_full   = tokenizer(full_text,   return_tensors="pt",
                           truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
    enc_prompt = tokenizer(prompt,      return_tensors="pt",
                           truncation=True, max_length=MAX_SEQ_LEN).to("cuda")
    plen = enc_prompt["input_ids"].shape[1]

    with torch.no_grad():
        logits = mdl(**enc_full).logits[0]        # (seq, vocab)

    log_probs  = torch.log_softmax(logits, dim=-1)  # (seq, vocab)
    comp_ids   = enc_full["input_ids"][0, plen:]    # completion token IDs

    if len(comp_ids) == 0:
        return torch.zeros(1, device="cuda")

    # log_probs at positions [plen-1 .. plen+len-2] predict tokens at [plen .. plen+len-1]
    pred_positions = log_probs[plen - 1 : plen - 1 + len(comp_ids)]  # (n, vocab)
    ml = min(len(pred_positions), len(comp_ids))
    return pred_positions[:ml].gather(1, comp_ids[:ml].unsqueeze(1)).squeeze(1)  # (ml,)


def run_episode(task: str, do_sample: bool = True):
    """
    Run one full episode. Returns list of step dicts:
      {prompt, completion, reward, done, final_score}
    """
    steps = []
    sid, obs = env_reset(task)
    done = False
    guard = 0
    while not done and guard < obs.get("max_steps", 10) + 3:
        guard += 1
        prompt = build_prompt(obs)
        completion = generate_text(prompt, do_sample=do_sample)
        active_role = obs.get("active_role", "support_agent")
        action = parse_action(completion, active_role)
        result = env_step(sid, action)
        reward     = result["reward"]["value"]
        done       = result["done"]
        final_score = result.get("final_score")
        steps.append({
            "prompt": prompt, "completion": completion,
            "reward": reward, "done": done, "final_score": final_score,
        })
        obs = result["observation"]
    return steps


def episode_score(steps: list) -> float:
    """Aggregate episode steps into a scalar reward for GRPO."""
    if not steps:
        return 0.0
    GAMMA, STEP_W, FINAL_W = 0.95, 0.30, 0.70
    n = len(steps)
    disc_sum   = sum(GAMMA**t * s["reward"] for t, s in enumerate(steps))
    normalizer = sum(GAMMA**t for t in range(n)) or 1.0
    step_avg   = disc_sum / normalizer
    final      = steps[-1].get("final_score") or 0.0
    return STEP_W * step_avg + FINAL_W * final


print("✅ Rollout utilities ready")

## 7️⃣ Baseline Evaluation (Before Training)

In [ ]:
FastLanguageModel.for_inference(model)
print(f"Running {N_EVAL} baseline episodes on '{TASK}'...")

baseline_scores = []
for i in range(N_EVAL):
    steps = run_episode(TASK, do_sample=False)
    s = episode_score(steps)
    baseline_scores.append(s)
    print(f"  Episode {i+1}: score={s:.3f}  steps={len(steps)}")

baseline_mean = sum(baseline_scores) / len(baseline_scores)
print(f"\n📊 Baseline mean score: {baseline_mean:.3f}")

## 8️⃣ GRPO Training Loop

**GRPO algorithm:**
1. Collect `GROUP_SIZE` episode rollouts
2. Compute group-normalised advantage: `A_i = (R_i - μ) / σ`
3. For each step in each episode, compute:
   - Policy ratio: `r = exp(log π_θ - log π_ref)`  ← key fix vs. naïve implementations
   - Clipped PPO loss + KL penalty
4. Gradient step

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

CLIP_EPS = 0.2
KL_COEF  = 0.04

model.train()
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LEARNING_RATE, weight_decay=0.01,
)
scheduler = CosineAnnealingLR(optimizer, T_max=TOTAL_STEPS, eta_min=LEARNING_RATE * 0.1)

train_rewards, train_losses, step_log = [], [], []


def grpo_advantages(rewards):
    if len(rewards) < 2:
        return [0.0] * len(rewards)
    mu    = sum(rewards) / len(rewards)
    sigma = (sum((r - mu)**2 for r in rewards) / len(rewards)) ** 0.5
    return [(r - mu) / (sigma + 1e-8) for r in rewards]


print(f"🚀 GRPO training — {TOTAL_STEPS} steps, group_size={GROUP_SIZE}")
print("=" * 60)
t0 = time.time()

for step in range(1, TOTAL_STEPS + 1):

    # ── 1. Collect rollouts ──────────────────────────────────────────────────
    model.eval()
    FastLanguageModel.for_inference(model)
    group = [run_episode(TASK, do_sample=True) for _ in range(GROUP_SIZE)]

    rewards    = [episode_score(ep) for ep in group]
    advantages = grpo_advantages(rewards)

    # ── 2. Compute GRPO loss ─────────────────────────────────────────────────
    model.train()
    total_loss   = torch.tensor(0.0, requires_grad=True, device="cuda")
    total_tokens = 0

    for ep_steps, adv in zip(group, advantages):
        if not ep_steps:
            continue
        adv_t = torch.tensor(float(adv), device="cuda")

        for s in ep_steps:
            if not s["completion"].strip():
                continue

            # Current policy log-probs (with grad)
            cur_lp = get_log_probs(model, s["prompt"], s["completion"])

            # Reference policy log-probs (no grad — frozen)
            with torch.no_grad():
                ref_lp = get_log_probs(ref_model, s["prompt"], s["completion"])

            ml = min(len(cur_lp), len(ref_lp))
            if ml == 0:
                continue

            # ── Policy ratio: exp(log π_θ - log π_ref) ──
            # This is the CORRECT ratio — comparing current policy to reference,
            # NOT to itself (which would always be 1.0).
            ratio   = (cur_lp[:ml] - ref_lp[:ml]).exp()
            clipped = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS)

            # PPO surrogate loss (negative because we maximise reward)
            pg_loss = -torch.min(ratio * adv_t, clipped * adv_t).mean()

            # KL penalty: keep policy close to reference
            kl_loss = (cur_lp[:ml] - ref_lp[:ml]).mean()

            step_loss = pg_loss + KL_COEF * kl_loss
            total_loss = total_loss + step_loss * ml
            total_tokens += ml

    # ── 3. Gradient update ───────────────────────────────────────────────────
    if total_tokens > 0:
        loss = total_loss / total_tokens
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            filter(lambda p: p.requires_grad, model.parameters()), 0.5
        )
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        loss_val = loss.item()
    else:
        optimizer.zero_grad()
        loss_val = 0.0

    mean_r = sum(rewards) / len(rewards) if rewards else 0.0
    train_rewards.append(mean_r)
    train_losses.append(loss_val)
    step_log.append(step)

    if step % 5 == 0 or step == 1:
        elapsed = time.time() - t0
        print(f"  [Step {step:3d}/{TOTAL_STEPS}]  loss={loss_val:.4f}  "
              f"reward={mean_r:.3f}  lr={optimizer.param_groups[0]['lr']:.2e}  "
              f"elapsed={elapsed:.0f}s")

print(f"\n✅ Training complete in {time.time()-t0:.0f}s")

## 9️⃣ Post-Training Evaluation

In [ ]:
FastLanguageModel.for_inference(model)
print(f"Running {N_EVAL} post-training episodes on '{TASK}'...")

trained_scores = []
for i in range(N_EVAL):
    steps = run_episode(TASK, do_sample=False)
    s = episode_score(steps)
    trained_scores.append(s)
    print(f"  Episode {i+1}: score={s:.3f}  steps={len(steps)}")

trained_mean = sum(trained_scores) / len(trained_scores)
print(f"\n📊 Post-training mean: {trained_mean:.3f}")
print(f"📈 Improvement:        {baseline_mean:.3f} → {trained_mean:.3f}  ({trained_mean - baseline_mean:+.3f})")

## 📊 Training Curves & Before/After

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"GRPO Training — {MODEL} on {TASK}", fontsize=13, fontweight="bold")

def smooth(vals, w=5):
    if len(vals) < w:
        return vals
    return np.convolve(vals, np.ones(w) / w, mode="valid").tolist()

# ── Reward curve ──────────────────────────────────────────────────────────────
axes[0].plot(step_log, train_rewards, alpha=0.25, color="steelblue", label="Raw")
s = smooth(train_rewards)
axes[0].plot(step_log[len(step_log)-len(s):], s, color="steelblue", lw=2, label="Smoothed")
axes[0].set(xlabel="Step", ylabel="Mean Episode Reward", title="🎯 Reward Curve")
axes[0].legend(); axes[0].grid(alpha=0.3)

# ── Loss curve ────────────────────────────────────────────────────────────────
axes[1].plot(step_log, train_losses, alpha=0.25, color="tomato", label="Raw")
sl = smooth(train_losses)
axes[1].plot(step_log[len(step_log)-len(sl):], sl, color="tomato", lw=2, label="Smoothed")
axes[1].set(xlabel="Step", ylabel="GRPO Loss", title="📉 Loss Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

# ── Before / After ────────────────────────────────────────────────────────────
x = np.arange(N_EVAL)
w = 0.35
axes[2].bar(x - w/2, baseline_scores, w, label="Before", color="#ff6b6b", alpha=0.85)
axes[2].bar(x + w/2, trained_scores,  w, label="After",  color="#51cf66", alpha=0.85)
axes[2].axhline(baseline_mean, color="red",   ls="--", alpha=0.6, label=f"Baseline avg {baseline_mean:.2f}")
axes[2].axhline(trained_mean,  color="green", ls="--", alpha=0.6, label=f"Trained avg  {trained_mean:.2f}")
axes[2].set(xlabel="Episode", ylabel="Score", title="📊 Before vs After")
axes[2].set_xticks(x); axes[2].set_xticklabels([f"Ep {i+1}" for i in range(N_EVAL)])
axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("📁 Saved to training_results.png")

## ✅ Summary

In [ ]:
print("=" * 60)
print("  TRAINING SUMMARY")
print("=" * 60)
print(f"  Model          : {MODEL}")
print(f"  Task           : {TASK}")
print(f"  Steps          : {TOTAL_STEPS}")
print(f"  Group size     : {GROUP_SIZE}")
print(f"  LoRA rank      : {LORA_R}")
print()
print(f"  Baseline score : {baseline_mean:.3f}")
print(f"  Trained score  : {trained_mean:.3f}")
delta_pct = 100 * (trained_mean - baseline_mean) / max(0.01, baseline_mean)
print(f"  Improvement    : {trained_mean - baseline_mean:+.3f}  ({delta_pct:+.1f}%)")
print()
print(f"  Final loss     : {train_losses[-1]:.4f}")
print(f"  Final reward   : {train_rewards[-1]:.3f}")
print("=" * 60)
print()
print("💡 To go further:")
print("   - Increase TOTAL_STEPS to 200+ for meaningful improvement")
print("   - Advance TASK to 'curriculum_supervisor' once score ≥ 0.65")
print("   - Use GROUP_SIZE=8 for more stable advantage estimates")

## 💾 Save Trained LoRA Adapter (Optional)

In [ ]:
# Uncomment to save locally
# model.save_pretrained("trained_lora_adapter")
# tokenizer.save_pretrained("trained_lora_adapter")
# print("✅ Adapter saved to trained_lora_adapter/")

# Uncomment to push to Hugging Face Hub
# model.push_to_hub("YOUR_HF_USERNAME/customer-support-grpo", token="YOUR_HF_TOKEN")
# tokenizer.push_to_hub("YOUR_HF_USERNAME/customer-support-grpo", token="YOUR_HF_TOKEN")

print("Uncomment the lines above to save your trained adapter.")